## Notebook 15 — Significance of the RF vs MLP Gap
**Project:** Machine Learning for High Performance Optical Sorting
**Author:** Mohamed Tawfeek
**Description:** McNemar's test on the paired test-set predictions of the saved RF and MLP, followed by a five-seed split-train-evaluate sweep to measure how much of the gap is split noise

The headline comparison in this project is a 1.58 point accuracy difference between the
Random Forest and the MLP on a 380-image test set, which is six images. This notebook asks
two separate questions about that number.

1. **Is the difference on this split statistically distinguishable from chance?**
   McNemar's exact test on the paired predictions of the two saved models.
2. **Is the difference stable across splits?** A full split, train and evaluate loop
   repeated at five seeds, reporting the seed-to-seed spread of each metric and of the
   MLP-minus-RF gap.

Part 1 loads the committed pickles. Part 2 deliberately does not: it trains fresh models
inside this notebook so that the seed genuinely varies the split as well as the model, and
nothing in `results/` is overwritten.

In [1]:
import json
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.contingency_tables import mcnemar

In [2]:
CLASSES = ['glass', 'paper', 'cardboard', 'plastic', 'metal', 'trash']
CLASS_ORDER = sorted(CLASSES)   # sklearn orders classes alphabetically
FEATURES_PATH = os.path.join('..', 'results', 'features')
RESULTS_PATH = os.path.join('..', 'results')

SEEDS = [42, 1, 7, 13, 99]      # 42 is the split every other notebook reports

In [3]:
all_features = np.load(os.path.join(FEATURES_PATH, 'features.npy'))
all_labels = np.load(os.path.join(FEATURES_PATH, 'labels.npy'))
print(f"Features shape: {all_features.shape}")
print(f"Labels shape: {all_labels.shape}")

Features shape: (2527, 122)
Labels shape: (2527,)


In [4]:
# Reproduces the stratified 70/15/15 split used in notebooks 04 and 05, parameterised by seed
# so the same code serves both the seed-42 comparison and the multi-seed sweep

def make_split(features, labels, seed):
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        features, labels,
        test_size=0.15,
        random_state=seed,
        stratify=labels
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=0.176,
        random_state=seed,
        stratify=y_train_val
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


X_train, X_val, X_test, y_train, y_val, y_test = make_split(all_features, all_labels, 42)
print(f"Training set:   {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

Training set:   1769 samples
Validation set: 378 samples
Test set:       380 samples


### Part 1 — McNemar's test on the committed models

McNemar's test is the right test here because the two models are evaluated on the *same*
test images. An unpaired test would throw away that pairing and ask a weaker question. The
test looks only at the discordant pairs: images one model gets right and the other gets
wrong. Images both models agree on, right or wrong, carry no information about which model
is better and are excluded by construction.

`exact=True` uses the binomial test rather than the chi-squared approximation, which is
the correct choice when the discordant counts are small.

In [5]:
with open(os.path.join(RESULTS_PATH, 'rf_final_model.pkl'), 'rb') as f:
    rf = pickle.load(f)

with open(os.path.join(RESULTS_PATH, 'mlp_final_model.pkl'), 'rb') as f:
    mlp = pickle.load(f)

with open(os.path.join(RESULTS_PATH, 'mlp_scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)

rf_pred = rf.predict(X_test)
mlp_pred = mlp.predict(scaler.transform(X_test))

print(f"RF  test accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"MLP test accuracy: {accuracy_score(y_test, mlp_pred):.4f}")
print(f"Difference:        {accuracy_score(y_test, mlp_pred) - accuracy_score(y_test, rf_pred):+.4f}"
      f"  ({int(round((accuracy_score(y_test, mlp_pred) - accuracy_score(y_test, rf_pred)) * len(y_test)))} images)")

# Sanity check that the reproduced split matches the one the pickles were evaluated on
saved_rf_cm = np.load(os.path.join(RESULTS_PATH, 'rf_confusion_matrix.npy'))
assert np.array_equal(confusion_matrix(y_test, rf_pred, labels=CLASS_ORDER), saved_rf_cm), \
    "Reproduced test partition does not match the committed RF confusion matrix"
print("\nSplit matches the committed rf_confusion_matrix.npy.")

RF  test accuracy: 0.7895
MLP test accuracy: 0.8053
Difference:        +0.0158  (6 images)

Split matches the committed rf_confusion_matrix.npy.


In [6]:
# Builds the 2x2 paired agreement table and runs McNemar's exact test on the discordant cells

rf_correct = (rf_pred == y_test)
mlp_correct = (mlp_pred == y_test)

n_both_wrong   = int(np.sum(~rf_correct & ~mlp_correct))
n_rf_only      = int(np.sum(rf_correct & ~mlp_correct))
n_mlp_only     = int(np.sum(~rf_correct & mlp_correct))
n_both_correct = int(np.sum(rf_correct & mlp_correct))

table = np.array([[n_both_correct, n_rf_only],
                  [n_mlp_only,     n_both_wrong]])

contingency = pd.DataFrame(
    table,
    index=['RF correct', 'RF wrong'],
    columns=['MLP correct', 'MLP wrong']
)

print("Paired agreement on the 380-image test set:\n")
print(contingency.to_string())
print(f"\nDiscordant pairs: {n_rf_only + n_mlp_only}"
      f"  (RF only: {n_rf_only}, MLP only: {n_mlp_only})")

Paired agreement on the 380-image test set:

            MLP correct  MLP wrong
RF correct          272         28
RF wrong             34         46

Discordant pairs: 62  (RF only: 28, MLP only: 34)


In [7]:
result = mcnemar(table, exact=True)

print("=== McNemar's exact test: RF vs MLP ===\n")
print(f"Test statistic (smaller discordant count): {result.statistic:.0f}")
print(f"p-value: {result.pvalue:.4f}")
print()

if result.pvalue < 0.05:
    print("p < 0.05: the two models make significantly different errors on this split.")
else:
    print("p >= 0.05: the difference between the two models on this split is not")
    print("distinguishable from the two classifiers being equally accurate.")

=== McNemar's exact test: RF vs MLP ===

Test statistic (smaller discordant count): 28
p-value: 0.5258

p >= 0.05: the difference between the two models on this split is not
distinguishable from the two classifiers being equally accurate.


### Part 2 — Five-seed split, train and evaluate sweep

McNemar's test answers a question about one split. It says nothing about how much the gap
would move if the 380 test images were drawn differently. That needs the whole pipeline
re-run.

Each seed re-draws the stratified 70/15/15 split *and* re-seeds both models, so the spread
below is the combined split-plus-initialisation variance a reader would see if they reran
the project with a different `random_state`. Hyperparameters are held at the values
notebooks 04 and 05 selected: they were tuned by cross-validation on the seed-42 training
partition, and re-tuning per seed would confound the measurement with the tuning
procedure.

In [8]:
# Macro Recovery is the composite TP/(TP+FP+FN) from notebook 08, averaged over classes.
# Recomputed here from the confusion matrix rather than imported, since notebook 08 defines
# it inline over a DataFrame and this loop only needs the macro value.

def macro_recovery(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    denominator = tp + fp + fn
    recovery = np.divide(tp, denominator, out=np.zeros_like(tp), where=denominator > 0)
    return recovery.mean()


def evaluate(y_true, y_pred):
    return {
        'accuracy':       accuracy_score(y_true, y_pred),
        'weighted_f1':    f1_score(y_true, y_pred, average='weighted'),
        'macro_f1':       f1_score(y_true, y_pred, average='macro'),
        'macro_recovery': macro_recovery(y_true, y_pred),
    }

In [9]:
# Full split -> train -> evaluate loop for both models at one seed. Fresh models throughout:
# nothing in results/ is read or written by this function.

def run_seed(seed):
    X_tr, _, X_te, y_tr, _, y_te = make_split(all_features, all_labels, seed)

    rf_seed = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=seed,
        n_jobs=-1
    )
    rf_seed.fit(X_tr, y_tr)
    rf_scores = evaluate(y_te, rf_seed.predict(X_te))

    scaler_seed = StandardScaler()
    X_tr_scaled = scaler_seed.fit_transform(X_tr)
    X_te_scaled = scaler_seed.transform(X_te)

    mlp_seed = MLPClassifier(
        hidden_layer_sizes=(200,),
        alpha=0.01,
        max_iter=1000,
        random_state=seed
    )
    mlp_seed.fit(X_tr_scaled, y_tr)
    mlp_scores = evaluate(y_te, mlp_seed.predict(X_te_scaled))

    return rf_scores, mlp_scores

In [10]:
rows = []

for seed in SEEDS:
    rf_scores, mlp_scores = run_seed(seed)
    for model, scores in (('Random Forest', rf_scores), ('MLP', mlp_scores)):
        rows.append({'seed': seed, 'model': model, **scores})
    print(f"seed={seed:3d}  RF acc {rf_scores['accuracy']:.4f}   "
          f"MLP acc {mlp_scores['accuracy']:.4f}   "
          f"gap {mlp_scores['accuracy'] - rf_scores['accuracy']:+.4f}")

seed_results = pd.DataFrame(rows)
print()
print(seed_results.to_string(index=False))

seed= 42  RF acc 0.7895   MLP acc 0.8053   gap +0.0158


seed=  1  RF acc 0.7763   MLP acc 0.7974   gap +0.0211


seed=  7  RF acc 0.7605   MLP acc 0.8000   gap +0.0395


seed= 13  RF acc 0.7947   MLP acc 0.7737   gap -0.0211


seed= 99  RF acc 0.7763   MLP acc 0.8263   gap +0.0500

 seed         model  accuracy  weighted_f1  macro_f1  macro_recovery
   42 Random Forest  0.789474     0.789810  0.774748        0.638090
   42           MLP  0.805263     0.805543  0.791396        0.660464
    1 Random Forest  0.776316     0.773939  0.748688        0.606544
    1           MLP  0.797368     0.796918  0.773294        0.636049
    7 Random Forest  0.760526     0.757497  0.729635        0.583255
    7           MLP  0.800000     0.799948  0.793078        0.660811
   13 Random Forest  0.794737     0.791930  0.771700        0.635216
   13           MLP  0.773684     0.773703  0.748863        0.606701
   99 Random Forest  0.776316     0.769843  0.726981        0.591992
   99           MLP  0.826316     0.826131  0.814876        0.692266


In [11]:
METRICS = ['accuracy', 'weighted_f1', 'macro_f1', 'macro_recovery']

summary = (
    seed_results
    .groupby('model')[METRICS]
    .agg(['mean', 'std'])
    .round(4)
)

print("=== Mean and standard deviation across seeds 42, 1, 7, 13, 99 ===\n")
print(summary.to_string())

=== Mean and standard deviation across seeds 42, 1, 7, 13, 99 ===

              accuracy         weighted_f1         macro_f1         macro_recovery        
                  mean     std        mean     std     mean     std           mean     std
model                                                                                     
MLP             0.8005  0.0188      0.8004  0.0188   0.7843  0.0247         0.6513  0.0319
Random Forest   0.7795  0.0133      0.7766  0.0144   0.7504  0.0225         0.6110  0.0249


In [12]:
wide = seed_results.pivot(index='seed', columns='model', values=METRICS)

print("=== MLP minus RF, per metric ===\n")
print(f"{'Metric':<16} {'mean gap':>10} {'std':>10} {'MLP wins':>10}")
print("-" * 50)

gap_summary = {}
for metric in METRICS:
    gaps = wide[(metric, 'MLP')] - wide[(metric, 'Random Forest')]
    wins = int((gaps > 0).sum())
    gap_summary[metric] = {
        'mean': float(gaps.mean()),
        'std': float(gaps.std(ddof=1)),
        'mlp_wins': wins,
        'n_seeds': len(SEEDS),
    }
    print(f"{metric:<16} {gaps.mean():>+10.4f} {gaps.std(ddof=1):>10.4f} {wins:>7d}/{len(SEEDS)}")

=== MLP minus RF, per metric ===

Metric             mean gap        std   MLP wins
--------------------------------------------------
accuracy            +0.0211     0.0273       4/5
weighted_f1         +0.0238     0.0284       4/5
macro_f1            +0.0340     0.0430       4/5
macro_recovery      +0.0402     0.0504       4/5


In [13]:
# Puts the seed-42 gap in units of the seed-to-seed spread, which is the number the README
# sentence needs: a gap much smaller than one standard deviation is not a model difference.

seed42_gap = (
    wide.loc[42, ('accuracy', 'MLP')] - wide.loc[42, ('accuracy', 'Random Forest')]
)
accuracy_gap_std = gap_summary['accuracy']['std']

print(f"Seed-42 accuracy gap:        {seed42_gap:+.4f}")
print(f"Seed-to-seed gap std:        {accuracy_gap_std:.4f}")
print(f"Gap in units of that std:    {seed42_gap / accuracy_gap_std:+.2f}")
print()
print(f"Test-set size:               {len(y_test)} images")
print(f"One standard deviation of the gap is worth "
      f"{accuracy_gap_std * len(y_test):.1f} images.")

Seed-42 accuracy gap:        +0.0158
Seed-to-seed gap std:        0.0273
Gap in units of that std:    +0.58

Test-set size:               380 images
One standard deviation of the gap is worth 10.4 images.


In [14]:
significance_results = {
    'mcnemar': {
        'split_seed': 42,
        'n_test': int(len(y_test)),
        'contingency': {
            'both_correct': n_both_correct,
            'rf_correct_mlp_wrong': n_rf_only,
            'mlp_correct_rf_wrong': n_mlp_only,
            'both_wrong': n_both_wrong,
        },
        'exact': True,
        'statistic': float(result.statistic),
        'p_value': float(result.pvalue),
    },
    'seed_sweep': {
        'seeds': SEEDS,
        'per_model': {
            model: {
                metric: {
                    'mean': float(group[metric].mean()),
                    'std': float(group[metric].std(ddof=1)),
                }
                for metric in METRICS
            }
            for model, group in seed_results.groupby('model')
        },
        'mlp_minus_rf': gap_summary,
    },
}

with open(os.path.join(RESULTS_PATH, 'significance_results.json'), 'w') as f:
    json.dump(significance_results, f, indent=2)

seed_results.to_csv(os.path.join(RESULTS_PATH, 'seed_sweep.csv'), index=False)

print("Saved results/significance_results.json")
print("Saved results/seed_sweep.csv")

Saved results/significance_results.json
Saved results/seed_sweep.csv


In [15]:
# Prints the sentence for the README results section so the claim and the numbers cannot drift

rf_acc = seed_results.query("model == 'Random Forest'")['accuracy']
mlp_acc = seed_results.query("model == 'MLP'")['accuracy']

print("--- README sentence ---\n")
print(
    f"McNemar's exact test on the paired test-set predictions gives p = {result.pvalue:.2f} "
    f"({n_rf_only} images the RF alone gets right against {n_mlp_only} for the MLP), and across "
    f"seeds 42, 1, 7, 13 and 99 accuracy is {rf_acc.mean():.4f} +/- {rf_acc.std(ddof=1):.4f} for the RF "
    f"and {mlp_acc.mean():.4f} +/- {mlp_acc.std(ddof=1):.4f} for the MLP, with the MLP ahead on "
    f"{gap_summary['accuracy']['mlp_wins']} of 5 seeds and a mean gap of "
    f"{gap_summary['accuracy']['mean']:+.4f} +/- {gap_summary['accuracy']['std']:.4f}."
)

--- README sentence ---

McNemar's exact test on the paired test-set predictions gives p = 0.53 (28 images the RF alone gets right against 34 for the MLP), and across seeds 42, 1, 7, 13 and 99 accuracy is 0.7795 +/- 0.0133 for the RF and 0.8005 +/- 0.0188 for the MLP, with the MLP ahead on 4 of 5 seeds and a mean gap of +0.0211 +/- 0.0273.
